# Tecan Infinite M1000 Pro

```{device-card} tecan-infinite-m1000-pro
```

| Property | Value |
| --- | --- |
| Connection | USB (libusbK driver on Windows) |
| Operations | absorbance (single wavelength, reference wavelength, spectrum, multiple reads per well), top fluorescence (fixed or optimal gain, optimal Z, flash modes, multiple reads, excitation/emission spectra), shaking, temperature, plate carrier |

The backend sends the same commands as Tecan's i-control 2.0. It was checked by replaying i-control's USB captures (identical commands, results equal to i-control's Excel export) and by running every operation on a reader with an empty carrier. It has not yet been compared with i-control on plates with real samples. See the [introduction](intro) for what is not implemented.

## Connection

- On Windows the reader needs the **libusbK** driver for PyLabRobot (install it once with Zadig). i-control needs Tecan's driver, so only one of them can use the reader at a time.
- Close i-control before connecting.
- The backend has no software safety limits beyond the reader's own firmware checks. Keep the carrier path clear and supervise runs.

## Setup

`setup()` connects and replays i-control's startup sequence, which initializes the reader's axes and monochromators. The reader reports its progress in the log.

In [ ]:
from pylabrobot.tecan import InfiniteM1000Pro

reader = InfiniteM1000Pro()
await reader.setup()

## Plate carrier

`open()` moves the carrier out so you can place a plate; `close()` moves it back in.

In [ ]:
await reader.open()

In [ ]:
await reader.close()

## Reader status

In [ ]:
await reader.request_instrument_status()

## Plates and plate definitions

Every read takes two plate arguments:

- `plate`: a PyLabRobot plate. It supplies the well names; pass `wells` to read a subset, or `None` for every well.
- `plate_definition`: the i-control plate type, which sets the well positions and the read height. The backend reads i-control's `.pdfx` files from `C:\Users\Public\Documents\Tecan\Pdfx` when they are installed; without them `GRE96ft` and `GRE384ft` are built in. You can also pass a `PlateGeometry`.

Use the plate definition that matches the physical plate in the reader.

In [ ]:
from pylabrobot.resources.biorad.plates import biorad_96_wellplate_200uL_Vb

plate = biorad_96_wellplate_200uL_Vb("plate")

## Absorbance

Read all wells at 600 nm. The read height is calculated from the plate definition the way i-control does it; pass `plate_lid=True` for a plate with a lid.

In [ ]:
absorbance = await reader.read_absorbance(plate, None, 600, plate_definition="GRE96ft")
absorbance.data["A1"]

With a reference wavelength, `data` holds the difference and `measurement` / `reference` hold both ODs:

In [ ]:
absorbance = await reader.read_absorbance(
  plate, plate.get_items("A1:H2"), 600, plate_definition="GRE96ft", reference_wavelength=900
)
absorbance.data["A1"], absorbance.measurement["A1"], absorbance.reference["A1"]

### Multiple reads per well

`reads_per_well=3` reads a 3 × 3 grid in each well. Patterns are `"filled_square"`, `"square"`, `"x_line"`, `"y_line"` and `"xy_lines"`; `multi_read_border` is the distance from the well edge in µm. `data` is the mean, `standard_deviation` and `reads` hold the details. i-control requires a settle time of 10–100 ms for multiple reads.

In [ ]:
absorbance = await reader.read_absorbance(
  plate, plate.get_items("A1:B2"), 600, plate_definition="GRE96ft",
  reads_per_well=3, settle_time=10,
)
absorbance.data["A1"], absorbance.standard_deviation["A1"]

### Absorbance spectrum

Results are keyed by well, then wavelength. A spectrum takes about 47 s per column position for a full 230–1000 nm scan.

In [ ]:
spectrum = await reader.read_absorbance_spectrum(
  plate, plate.get_items(["A1", "A2"]), start_wavelength=400, end_wavelength=700,
  step_size=10, plate_definition="GRE96ft",
)
spectrum.data["A1"]

## Fluorescence (top)

Excitation 485 nm, emission 535 nm, gain 100. Bandwidths, flashes, flash mode (`400`, `100` or `"on_the_fly"` Hz), integration and lag time, settle time and Z (µm) are parameters.

In [ ]:
fluorescence = await reader.read_fluorescence(
  plate, plate.get_items("A1:B3"), 485, 535, plate_definition="GRE96ft",
  excitation_bandwidth=9, emission_bandwidth=20, gain=100, flashes=25,
)
fluorescence.data["A1"]

### Optimal gain and Z

`gain="optimal"` lets the reader find the gain as i-control does, from all selected wells or from `gain_well`; `optimal_gain_percentage` sets the target. `z="optimal"` finds the read height in `z_well` (default: the first well). The result reports the values used.

In [ ]:
fluorescence = await reader.read_fluorescence(
  plate, plate.get_items("A1:B3"), 485, 535, plate_definition="GRE96ft",
  excitation_bandwidth=9, emission_bandwidth=20, gain="optimal", gain_well=plate.get_item("A1"),
)
fluorescence.gain

In [ ]:
fluorescence = await reader.read_fluorescence(
  plate, plate.get_items("A1:B3"), 485, 535, plate_definition="GRE96ft",
  excitation_bandwidth=9, emission_bandwidth=20, gain=100, z="optimal",
)
fluorescence.z

### Fluorescence spectra

Scan the excitation (fixed emission) or the emission (fixed excitation). Values are corrected with the reader's correction tables, as in i-control.

In [ ]:
emission = await reader.read_fluorescence_spectrum(
  plate, plate.get_items(["A1", "A2"]), scan="emission", start_wavelength=450,
  end_wavelength=550, step_size=5, excitation_wavelength=350, plate_definition="GRE96ft",
)
emission.data["A1"]

## Stopping a measurement

If a measurement is cancelled (for example by interrupting the kernel) or fails, the backend stops the reader the way i-control's Stop button does. The reader finishes its current step first, which can take up to about a minute at the start of a long spectrum, and is then ready for the next read.

## Shaking

Shake for a number of seconds. Modes are `"linear"`, `"orbital"` and `"double"`; the amplitude is in mm and must be one of i-control's values (see `SHAKING_FREQUENCIES` in the backend).

In [ ]:
await reader.shake(5, mode="orbital", amplitude=2.0)

## Temperature

In [ ]:
await reader.request_current_temperature()

In [ ]:
await reader.set_temperature(30.0)

In [ ]:
await reader.stop_temperature_control()

## Disconnect

Disconnecting does not move the carrier.

In [ ]:
await reader.stop()